## Load Config 

In [1]:
import sys
from pathlib import Path

import os
from dotenv import load_dotenv
import json
sys.path.insert(0, '..')
# Load variables from .env into the environment
load_dotenv()

# Access the variables
api_key = os.getenv("OPENAI_API_KEY") 
from utils.config_loader import load_config
config = load_config('../config/config.yaml')
print(config)

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from core.state import PipelineState
from agents.watchlist_agent import WatchlistContextAgent
from agents.retrieval_agent import NewsRetrievalAgent
from agents.market_data_agent import MarketDataAgent
from agents.filter_agent import NoiseFilterAgent
from agents.filter_critic_agent import FilterCriticAgent
from agents.clustering_agent import EventClusteringAgent
from agents.summarization_agent import ImpactSummarizationAgent
from agents.ranking_agent import ImportanceRankingAgent
from agents.notification_agent import NotificationAgent

{'simulation_mode': True, 'retrieval': {'lookback_hours': 720, 'sources': {'sgx_announcements': True, 'newsapi': True, 'yahoo_finance': True, 'cna': True, 'straits_times': True, 'mas': True, 'sbr': True, 'reuters': True, 'nikkei_asia': True, 'cnbc_asia': True, 'reddit': False}, 'max_articles_per_query': 20, 'fetch_full_content': True}, 'newsapi_key': 'b534ae4e571a48d285f602636c3bc4e2', 'newsapi_page_size': 20, 'sgx_request_delay': 1.5, 'sgx_max_pages': 10, 'retrieval_max_workers': 8, 'filtering': {'min_snippet_length': 20, 'similarity_threshold': 0.85, 'llm_model': 'gpt-4o', 'llm_enabled': True}, 'clustering': {'method': 'agglomerative', 'min_cluster_size': 1, 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'}, 'summarization': {'llm_model': 'gpt-4o', 'max_tokens': 512, 'temperature': 0.2, 'verification_enabled': True}, 'ranking': {'weights': {'event_type': 0.4, 'corroboration': 0.25, 'novelty': 0.2, 'credibility': 0.15}, 'source_credibility': {'tier_1': ['SGX Announcements',

In [2]:
initial = PipelineState()
#config['filtering']['llm_enabled'] = False
config['simulation_mode'] = True
watchlist_agent = WatchlistContextAgent(config)
market_data_agent = MarketDataAgent(config)
retrieval_agent = NewsRetrievalAgent(config)
filter_agent = NoiseFilterAgent(config)
filter_critic_agent = FilterCriticAgent(config)
event_clustering_agent = EventClusteringAgent(config)
summarization_agent = ImpactSummarizationAgent(config)
ranking_agent = ImportanceRankingAgent(config)
notification_agent = NotificationAgent(config)

## Agent Watchlist Test

In [3]:
initial.watchlist = ["AAPL", "GOOGL", "AMZN"]
wa_agent_result = watchlist_agent.run(initial)

[2026-04-08 11:44:28] INFO WatchlistContextAgent — [WatchlistContextAgent] Starting. 3 tickers: ['AAPL', 'GOOGL', 'AMZN']
[2026-04-08 11:44:28] INFO WatchlistContextAgent — [WatchlistContextAgent] Done. 3 bundles produced


In [4]:
wa_agent_result.to_json("watchlist_test_output.json")

## Agent Watchlist Load

In [5]:
with open("watchlist_test_output.json", "r") as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
wa_agent_result = PipelineState(**data)

## Retrieval Agent

In [6]:
re_agent_result =  retrieval_agent.run(wa_agent_result) 
re_agent_result

[2026-04-08 11:44:29] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Starting. Fetching from 5 sources for 3 tickers
[2026-04-08 11:44:29] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Done. [Agent 2] ✓ 226 articles (simulated)


PipelineState(watchlist=['AAPL', 'GOOGL', 'AMZN'], query_bundles=[{'ticker': 'AAPL', 'company_name': 'Apple Inc.', 'aliases': ['Apple', 'Apple Inc'], 'industry': 'Consumer Electronics', 'company_queries': ['Apple earnings Q4 2025', 'Apple iPhone sales', 'Apple stock news'], 'industry_queries': ['smartphone market trends 2025', 'consumer electronics industry news']}, {'ticker': 'GOOGL', 'company_name': 'Alphabet Inc.', 'aliases': ['Google', 'Alphabet'], 'industry': 'Internet Services', 'company_queries': ['Google earnings Q4 2025', 'Google search engine updates', 'Alphabet stock news'], 'industry_queries': ['internet services market trends 2025', 'tech industry news']}, {'ticker': 'AMZN', 'company_name': 'Amazon.com, Inc.', 'aliases': ['Amazon', 'Amazon.com'], 'industry': 'E-commerce', 'company_queries': ['Amazon earnings Q4 2025', 'Amazon Prime subscription growth', 'Amazon stock news'], 'industry_queries': ['e-commerce market trends 2025', 'retail industry news']}], raw_articles=[{'ti

In [7]:
re_agent_result.to_json("retrieval_test_output.json")

## Retrieval Agent Load

In [8]:
with open("retrieval_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
re_agent_result = PipelineState(**data)

## Market Retrieval Agent

In [9]:
mr_agent_result =  market_data_agent.run(re_agent_result.watchlist) 
re_agent_result.market_context = mr_agent_result

[2026-04-08 11:44:30] INFO MarketDataAgent — [MarketDataAgent] Starting. Fetching market data for 3 tickers.
[2026-04-08 11:44:30] INFO MarketDataAgent — [MarketDataAgent] Done. Simulation mode: Loaded market context for 3 tickers.


In [10]:
re_agent_result.to_json("market_data_test_output.json")

## Load Market Retrieval Agent

In [11]:
with open("market_data_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
re_agent_result = PipelineState(**data)

## News Filtering Agent

In [12]:
filter_agent_result = filter_agent.run(re_agent_result)

[2026-04-08 11:44:33] INFO NoiseFilterAgent — [NoiseFilterAgent] Starting. 226 raw articles
[2026-04-08 11:44:33] INFO NoiseFilterAgent — [NoiseFilterAgent] Done. [Agent 3] ✓ 26 articles after filtering (simulated)


In [13]:
filter_agent_result.to_json("filter_test_output.json")

## Load News Filter Agent 

In [14]:
with open("filter_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
filter_agent_result = PipelineState(**data)

## Filter Critic Agent

In [15]:
filter_critic_result = filter_critic_agent.run(filter_agent_result)

[2026-04-08 11:44:39] INFO FilterCriticAgent — [FilterCriticAgent] Starting. Evaluating 26 filtered articles (attempt 2/3)
[2026-04-08 11:44:39] INFO FilterCriticAgent — [FilterCriticAgent] Done. [Agent 3] ✓ 26 no critic issues (simulated)


In [16]:
filter_critic_result.to_json("filter_critic_test_output.json")

## Load Filter Critic Agent

In [17]:
with open("filter_critic_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
filter_critic_result = PipelineState(**data)

## Clustering Agent

In [19]:
clustering_result = event_clustering_agent.run(filter_critic_result)

[2026-04-08 11:45:18] INFO EventClusteringAgent — [EventClusteringAgent] Starting. 26 clean articles
[2026-04-08 11:45:18] INFO EventClusteringAgent — [EventClusteringAgent] Done. [Agent 4] ✓ Formed 12 event clusters (simulated)


In [20]:
clustering_result.to_json("clustring_test_output.json")

## Load Clustering Agent

In [21]:
with open("clustring_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
clustering_result = PipelineState(**data)

## Summarization Agent

In [22]:
Summarization_result = summarization_agent.run(clustering_result)

[2026-04-08 11:45:24] INFO ImpactSummarizationAgent — [ImpactSummarizationAgent] Starting. 12 event clusters
[2026-04-08 11:45:24] INFO ImpactSummarizationAgent — [ImpactSummarizationAgent] Done. [Agent 5] ✓ Generated 12 event cards (simulated)


In [23]:
Summarization_result.to_json("summarization_test_output.json")

## Load Summarization Result

In [24]:
with open("summarization_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
Summarization_result = PipelineState(**data)

## Ranking Agent

In [25]:
ranking_result = ranking_agent.run(Summarization_result)

[2026-04-08 11:45:31] INFO ImportanceRankingAgent — [ImportanceRankingAgent] Starting. 12 event cards
[2026-04-08 11:45:31] INFO ImportanceRankingAgent — [ImportanceRankingAgent] Done. [Agent 6] ✓ Ranked 12 events (simulated)


In [26]:
ranking_result.to_json("ranking_test_output.json")

## Load Ranking Agent

In [27]:
with open("ranking_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
ranking_result = PipelineState(**data)

## Notification Agent

In [28]:
notification_result = notification_agent.run(ranking_result)


[2026-04-08 11:45:38] INFO NotificationAgent — [NotificationAgent] Starting. 12 ranked events
[2026-04-08 11:45:38] INFO NotificationAgent — HTML digest saved to output\digest.html
[2026-04-08 11:45:38] INFO NotificationAgent — [NotificationAgent] Done. [Agent 7] ✓ Digest ready — 1 High / 7 Medium / 4 Low


# Load Ranking Agent

In [29]:
from langgraph.graph import StateGraph, END
from core.state import PipelineState

# Import your agent wrappers
from agents.watchlist_agent import watchlist_agent
from agents.retrieval_agent import retrieval_agent
from agents.filter_agent import filter_agent
from agents.filter_critic_agent import filter_critic_agent, route_after_filter_critic

# --- 1. Load the Config Once ---
config = load_config('../config/config.yaml')
config['simulation_mode'] = True
#config['filtering']['llm_enabled'] = False

def watchlist_node(state: PipelineState):
    from agents.watchlist_agent import WatchlistContextAgent
    # Combine YAML config with environment keys
    agent = WatchlistContextAgent({**config, "openai_key": os.getenv("OPENAI_API_KEY")})
    return agent.run(state)

def retrieval_node(state: PipelineState):
    from agents.retrieval_agent import NewsRetrievalAgent
    agent = NewsRetrievalAgent(config)
    return agent.run(state)

def filter_node(state: PipelineState):
    from agents.filter_agent import NoiseFilterAgent
    agent = NoiseFilterAgent(config)
    return agent.run(state)

def critic_node(state: PipelineState):
    from agents.filter_critic_agent import FilterCriticAgent
    agent = FilterCriticAgent(config.get("filtering", {}))
    return agent.run(state)

def build_financial_news_pipeline():
    # 1. Initialize the Graph with your Dataclass
    workflow = StateGraph(PipelineState)

    # 2. Add Nodes
    workflow.add_node("resolve_watchlist", watchlist_node)
    workflow.add_node("fetch_news", retrieval_node)
    workflow.add_node("filter_noise", filter_node)
    workflow.add_node("critic_eval", critic_node)
    
    # Placeholder for the next agent in your pipeline (Agent 4)
    # workflow.add_node("cluster_events", event_clustering_agent)

    # 3. Define the Flow (Edges)
    workflow.set_entry_point("resolve_watchlist")
    
    workflow.add_edge("resolve_watchlist", "fetch_news")
    workflow.add_edge("fetch_news", "filter_noise")
    workflow.add_edge("filter_noise", "critic_eval")

    # 4. Add Conditional Routing (The Loop)
    workflow.add_conditional_edges(
        "critic_eval",
        route_after_filter_critic,
        {
            "retry_filter": "filter_noise",  # Loop back to Agent 3
            "continue": END,                # Or link to "cluster_events"
            "abort": END                    # End on critical error
        }
    )

    return workflow.compile()


In [30]:
app = build_financial_news_pipeline()
    
initial_state = PipelineState(watchlist=["TSLA", "NVDA", "AAPL"])
final_state = app.invoke(initial_state)
    
print(f"Final Article Count: {final_state['clean_article_count']}")
print(f"Retries attempted: {final_state['filter_retry_count']}")

[2026-04-08 11:58:51] INFO WatchlistContextAgent — [WatchlistContextAgent] Starting. 3 tickers: ['TSLA', 'NVDA', 'AAPL']
[2026-04-08 11:58:51] INFO WatchlistContextAgent — [WatchlistContextAgent] Done. 3 bundles produced
[2026-04-08 11:58:51] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Starting. Fetching from 5 sources for 3 tickers
[2026-04-08 11:58:51] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Done. [Agent 2] ✓ 226 articles (simulated)
[2026-04-08 11:58:51] INFO NoiseFilterAgent — [NoiseFilterAgent] Starting. 226 raw articles
[2026-04-08 11:58:51] INFO NoiseFilterAgent — [NoiseFilterAgent] Done. [Agent 3] ✓ 26 articles after filtering (simulated)
[2026-04-08 11:58:51] INFO FilterCriticAgent — [FilterCriticAgent] Starting. Evaluating 26 filtered articles (attempt 2/3)
[2026-04-08 11:58:51] INFO FilterCriticAgent — [FilterCriticAgent] Done. [Agent 3] ✓ 26 no critic issues (simulated)
Final Article Count: 26
Retries attempted: 1


In [19]:
final_state['clean_article_count']

36